# Cortex Colab Sync

Notebook de travail pour:
- charger un plan d'entrainement curé
- verifier les items acceptés
- produire un payload signé pour Cortex
- pousser le résultat vers `cortex-orchestrator`

Ce notebook reste advisory-first: il n'accorde aucun droit d'exécution.

In [ ]:
import json, hmac, hashlib, urllib.request
from pathlib import Path

PLAN_PATH = Path('internal-plan.json')
PAYLOAD_PATH = Path('verified-colab-result.json')
CORTEX_URL = 'http://127.0.0.1:18081/v1/training/colab/ingest'
CORTEX_SHARED_SECRET = 'REPLACE_ME'


In [ ]:
plan = json.loads(PLAN_PATH.read_text(encoding='utf-8'))
accepted = plan.get('accepted', [])
skipped_known = plan.get('skipped_known', [])
rejected = plan.get('rejected', [])

print('accepted:', len(accepted))
print('skipped_known:', len(skipped_known))
print('rejected:', len(rejected))
accepted[:3]

In [ ]:
target_agents = sorted({agent for item in accepted for agent in item.get('assigned_agents', [])})
payload = {
    'source': 'google_colab',
    'run_id': 'colab-run-manual',
    'training_plan_id': 'colab-plan-manual',
    'target_agents': target_agents,
    'dataset_fingerprint': plan.get('stats', {}).get('submitted', 0) and 'replace-with-real-fingerprint' or 'empty-plan',
    'knowledge_registry_fingerprint': 'replace-with-known-registry-fingerprint',
    'accepted_item_ids': [item['sample_id'] for item in accepted],
    'verification': {
        'status': 'verified',
        'novelty_gate_applied': True,
        'offensive_content_filtered': True,
        'known_attack_filter_applied': True,
        'human_reviewed': True,
        'accepted_count': len(accepted),
        'skipped_known_count': len(skipped_known),
        'rejected_count': len(rejected),
        'reviewer': 'analyst@cortex.local',
        'notes': 'Prepared from curated plan in Colab.'
    }
}
PAYLOAD_PATH.write_text(json.dumps(payload, indent=2), encoding='utf-8')
payload

In [ ]:
canonical = json.dumps(payload, sort_keys=True, separators=(',', ':')).encode('utf-8')
signature = hmac.new(CORTEX_SHARED_SECRET.encode('utf-8'), canonical, hashlib.sha256).hexdigest()
request = urllib.request.Request(
    CORTEX_URL,
    data=canonical,
    headers={
        'Content-Type': 'application/json',
        'x-cortex-colab-signature': signature,
    },
    method='POST',
)
with urllib.request.urlopen(request, timeout=30) as response:
    print(response.status)
    print(response.read().decode('utf-8'))
